# S0 · Cuaderno de muestra · ¿Cada cuánto registrar?
**Sistemas de adquisición de datos · TSU en Electrónica y Pruebas · Jabil**

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ggt13/DAQ_TSU/blob/main/plantillas/S0_AC_CLASE_Colab_PLANTILLA.ipynb)

## Objetivo
Decidir con números la **resolución** y el **intervalo de registro** de un registrador de temperatura, y
comprobar con datos reales que el error de la cuantización queda dentro de lo que predice la teoría.

## La situación
En la planta se quiere vigilar la temperatura de una línea con un registrador como el Elitech RC-5. Antes de dejarlo
grabando hay que contestar tres preguntas: ¿qué tan fino mide?, ¿cuánto dura su memoria? y ¿cada cuánto debe
registrar para no perder lo que pasa en el proceso? Cada alumno tiene un registrador y un proceso distintos: los
datos salen de su código.

## En qué consiste este cuaderno
| Parte | Tiempo | Qué haces | Qué produce |
|---|---|---|---|
| **A · Actividad en clase** | 30 min | Cuatro ejercicios (★ primero). En cada uno **escribes** la expresión en Python, la sustitución con tus números y su unidad, y lo que predice para la práctica | Tus cuatro resultados y tus predicciones |
| **B · Práctica** | 80 min | Tomas una semana de temperatura real de Guadalajara, la registras con **tu** convertidor y mides el error | Una gráfica firmada, tu conclusión y el archivo `S0_<código>.csv` |
| **Hoja de resultados** | en la práctica | Importas el CSV en tu hoja de cálculo de Google; la tabla y la gráfica se arman solas; contestas las preguntas | El reporte que entregas |

## Qué entregas y cómo se califica
1. **Este cuaderno**, al terminar la sesión, con todo ejecutado (Entorno de ejecución → Reiniciar y ejecutar todo).
   ★ 3 pts · ejercicios 2, 3 y 4, 2 pts cada uno · cierre 1 pt. Se califica que el resultado sea correcto para
   **tus** datos y que la sustitución y la predicción sean tuyas.
2. **La hoja de resultados** con el reporte, en 7 días naturales.

## Reglas
- **Archivo → Guardar una copia en Drive** antes de empezar; trabaja en tu copia.
- Completa sólo lo marcado con ✎. Ninguna celda avanza si falta algo: el mensaje dice qué.
- La revisión automática dice si el ejercicio está **completo**, no si está **bien**; eso lo califica el profesor.
- Si usas un asistente de IA, dilo en tu reporte. Las sustituciones y las respuestas deben llevar **tus** números.

In [ ]:
#@title Herramientas del curso (no modificar)
"""Herramientas comunes del curso «Sistemas de adquisición de datos» 2027.

Identificación del alumno, datos personales a partir de su código, firma de las gráficas,
huella de los resultados y CSV firmado para la hoja de resultados.

Este archivo es la fuente única: los cuadernos de cada sesión copian su contenido en la celda
«Herramientas del curso». Si se corrige aquí, se regeneran los cuadernos.
"""
import csv
import datetime as _dt
import hashlib
import zlib

import numpy as np

try:
    from zoneinfo import ZoneInfo
    _ZONA = ZoneInfo("America/Mexico_City")
except Exception:          # Python sin zoneinfo: se usa la hora del sistema
    _ZONA = None

_VALORES_DE_EJEMPLO = {"", "TU NOMBRE", "NOMBRE APELLIDO", "000000000", "123456789"}


class Alumno:
    """Datos de quien ejecuta el cuaderno. Se crea con identificar()."""

    def __init__(self, nombre, codigo, sesion):
        self.nombre = nombre
        self.codigo = codigo
        self.sesion = sesion

    @property
    def semilla(self):
        return int(self.codigo) % (2 ** 32)

    def rng(self, etiqueta=""):
        """Generador de números aleatorios propio del alumno.

        Cada etiqueta da una secuencia independiente: agregar un parámetro nuevo
        no cambia los que ya existían.
        """
        return np.random.default_rng([self.semilla, zlib.crc32(etiqueta.encode("utf-8"))])

    def __repr__(self):
        return f"Alumno({self.nombre!r}, {self.codigo!r}, {self.sesion!r})"


def identificar(nombre, codigo, sesion):
    """Valida la celda de identificación y devuelve el Alumno.

    Detiene el cuaderno con un mensaje claro si el nombre o el código faltan
    o se dejaron con el valor de ejemplo.
    """
    nombre = " ".join(str(nombre).split())
    codigo = str(codigo).strip()
    if nombre.upper() in _VALORES_DE_EJEMPLO or len(nombre.split()) < 2:
        raise ValueError("Escribe tu nombre completo (nombre y apellido) en NOMBRE y vuelve a ejecutar.")
    if codigo in _VALORES_DE_EJEMPLO or not codigo.isdigit() or not 7 <= len(codigo) <= 10:
        raise ValueError("Escribe tu código de alumno (sólo dígitos, de 7 a 10) en CODIGO y vuelve a ejecutar.")
    return Alumno(nombre, codigo, sesion)


def parametro(alumno, etiqueta, minimo, maximo, decimales=3):
    """Un valor personal entre minimo y maximo, siempre el mismo para el mismo código."""
    valor = alumno.rng(etiqueta).uniform(minimo, maximo)
    return round(float(valor), decimales)


def asignar(alumno, opciones, salto=0):
    """Elige una opción de la lista según el código (estación, ventana de fechas, carga del motor...)."""
    opciones = list(opciones)
    return opciones[(int(alumno.codigo) + salto) % len(opciones)]


def _redondea(valor):
    if isinstance(valor, (int, float, np.floating, np.integer)):
        return format(float(valor), ".4g")
    return str(valor)


def huella(alumno, resultados):
    """Seis caracteres que dependen del código, la sesión y los resultados (redondeados a 4 cifras)."""
    partes = [alumno.codigo, alumno.sesion] + [f"{k}={_redondea(v)}" for k, v in sorted(resultados.items())]
    return hashlib.sha256("|".join(partes).encode("utf-8")).hexdigest()[:6].upper()


def ahora():
    momento = _dt.datetime.now(_ZONA) if _ZONA else _dt.datetime.now()
    return momento.strftime("%Y-%m-%d %H:%M")


def titulo(ax, texto, alumno):
    """Título de la gráfica con el nombre del alumno."""
    ax.set_title(f"{texto} · {alumno.nombre}")


def firma(fig, alumno, resultados=None):
    """Pie de la figura: nombre · código · sesión · fecha y hora · huella.

    Toda gráfica que se entrega debe pasar por aquí.
    """
    texto = f"{alumno.nombre} · {alumno.codigo} · {alumno.sesion} · {ahora()}"
    if resultados:
        texto += f" · huella {huella(alumno, resultados)}"
    fig.text(0.01, 0.005, texto, fontsize=8, color="0.35", ha="left", va="bottom")
    fig.subplots_adjust(bottom=max(fig.subplotpars.bottom, 0.14))
    fig._firmada = True
    return fig


def exportar_csv(ruta, alumno, resultados, columnas, extra=None):
    """Escribe el CSV firmado que se importa en la pestaña «CSV» de la hoja de resultados.

    Renglones de metadatos: nombre, codigo, sesion, huella, ejecutado y los de `extra` (p. ej. la fuente
    de los datos). Después, un renglón por resultado («resultado:<nombre>», valor). Luego un renglón vacío
    y la tabla: encabezados y datos. `columnas` es un diccionario {encabezado: lista de valores}, todas de
    la misma longitud. La hoja busca cada dato por su etiqueta, no por su renglón.
    """
    largos = {len(v) for v in columnas.values()}
    if len(largos) != 1:
        raise ValueError("Todas las columnas de la tabla deben tener el mismo número de renglones.")
    with open(ruta, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["nombre", alumno.nombre])
        w.writerow(["codigo", alumno.codigo])
        w.writerow(["sesion", alumno.sesion])
        w.writerow(["huella", huella(alumno, resultados)])
        w.writerow(["ejecutado", ahora()])
        for k, v in (extra or {}).items():
            w.writerow([k, v])
        for k, v in resultados.items():
            w.writerow([f"resultado:{k}", _redondea(v)])
        w.writerow([])
        w.writerow(list(columnas))
        for fila in zip(*columnas.values()):
            w.writerow([_redondea(x) for x in fila])
    return ruta


def leer_csv_firmado(ruta):
    """Para el profesor: devuelve (metadatos, resultados, tabla) de un CSV firmado."""
    meta, resultados, tabla = {}, {}, []
    with open(ruta, newline="", encoding="utf-8") as f:
        filas = list(csv.reader(f))
    i = 0
    while i < len(filas) and filas[i]:
        clave, valor = filas[i][0], filas[i][1]
        if clave.startswith("resultado:"):
            resultados[clave.split(":", 1)[1]] = float(valor)
        else:
            meta[clave] = valor
        i += 1
    tabla = filas[i + 1:] if i + 1 < len(filas) else []
    return meta, resultados, tabla


def verificar_csv(ruta):
    """Para el profesor: True si la huella del CSV corresponde a su código, sesión y resultados."""
    meta, resultados, _ = leer_csv_firmado(ruta)
    alumno = Alumno(meta["nombre"], meta["codigo"], meta["sesion"])
    return huella(alumno, resultados) == meta["huella"]


# ---------------------------------------------------------------------------------------------
# Revisión de lo que escribe el alumno. Comprueba que cada ejercicio esté COMPLETO, no que esté
# bien: el resultado correcto lo califica el profesor con la clave. Si falta algo, detiene el
# cuaderno y dice qué falta.
# ---------------------------------------------------------------------------------------------
import math
import re

REGISTRO = {}


def _normaliza(texto):
    t = str(texto).replace("\u202f", "").replace("\u00a0", "").replace(" ", "")
    t = re.sub(r"(?<![\d.])([1-9]\d{0,2})((?:,\d{3})+)(?![\d])",  # separador de miles: 32,000 -> 32000
               lambda m: m.group(1) + m.group(2).replace(",", ""), t)
    return t.replace(",", ".")                                       # coma decimal: 0,068 -> 0.068


def _formas(x):
    """Maneras razonables de escribir un número: entero, o redondeado a 2, 3 o 4 cifras significativas."""
    x = float(x)
    formas = set()
    if x == int(x) and abs(x) < 1e9:
        formas.add(str(int(x)))
    if abs(x) >= 100:
        formas.add(str(int(round(x))))                       # 34271.6 escrito como 34272
    if abs(x) >= 1:
        for dec in (1, 2):
            formas.add(f"{x:.{dec}f}".rstrip("0").rstrip("."))  # 123456.7 con sus decimales
    for cifras in (2, 3, 4, 5, 6):
        g = f"{x:.{cifras}g}"
        if "e" not in g:
            formas.add(g)
    return {f for f in formas if f not in ("", "-0")}


def aparece(x, texto):
    """True si el número x está escrito en el texto (sin confundir 0.03 con 0.0337)."""
    t = _normaliza(texto)
    def patron(f):
        cola = r"0*" if "." in f else r"(?:\.0*)?"            # 0.0342 = 0.034200 · 120 = 120.0, pero no 1200
        return r"(?<![\d.])" + re.escape(f) + cola + r"(?![\d])"
    return any(re.search(patron(f), t) for f in _formas(x))


def _palabras(texto):
    return len(re.findall(r"\w+", str(texto)))


def revisa(n, valor, sustitucion, prediccion, datos, unidades):
    """Revisión de un ejercicio de la parte A.

    valor: lo que calculó la expresión del alumno. sustitucion: texto con sus números, el resultado y la
    unidad (p. ej. «70 / 2**8 = 0.273 °C»). prediccion: qué espera ver en la práctica, con sus palabras.
    datos: los números personales que deben aparecer en la sustitución. unidades: unidades aceptadas.
    """
    faltan = []
    if valor is None:
        faltan.append("la expresión en Python")
    elif not isinstance(valor, (int, float, np.integer, np.floating)) or not math.isfinite(float(valor)):
        faltan.append("una expresión que dé un número")
    s = str(sustitucion)
    if "=" not in s:
        faltan.append("la sustitución con «=»")
    for d in datos:
        if not aparece(d, s):
            faltan.append(f"tu dato {d:g} en la sustitución")
    if valor is not None and not faltan and not aparece(valor, s.split("=")[-1]):
        faltan.append("el resultado que dio tu expresión, después del último «=»")
    if not any(u in s for u in unidades):
        faltan.append("la unidad (" + " o ".join(unidades) + ")")
    if _palabras(prediccion) < 10:
        faltan.append("lo que predice, en al menos 10 palabras")
    REGISTRO[n] = not faltan
    if faltan:
        raise ValueError(f"Ejercicio {n}: falta " + "; ".join(faltan) + ".")
    print(f"✔ Ejercicio {n} completo.")


def revisa_texto(nombre, texto, numeros, minimo=20):
    """Revisión de una respuesta escrita: largo mínimo y los números propios que debe citar.

    numeros: diccionario {qué es: valor}; el mensaje nombra lo que falta sin revelar el valor.
    """
    faltan = [que for que, x in numeros.items() if not aparece(x, texto)]
    if _palabras(texto) < minimo:
        faltan.insert(0, f"al menos {minimo} palabras")
    REGISTRO[nombre] = not faltan
    if faltan:
        raise ValueError(f"{nombre}: falta " + "; ".join(faltan) + ".")
    print(f"✔ {nombre} completo.")


import matplotlib.pyplot as plt

def completa(valor, que):
    if valor is None:
        raise ValueError(f'Completa {que} (✎) y vuelve a ejecutar.')
    return valor

print('Herramientas del curso listas.')

## Identificación
Tu nombre y tu código aparecen en cada gráfica y en el CSV. Sin ellos el cuaderno no avanza.

In [ ]:
NOMBRE = ""   # ✎ tu nombre completo
CODIGO = ""   # ✎ tu código de alumno
SESION = "S0"

alumno = identificar(NOMBRE, CODIGO, SESION)
print("Cuaderno de", alumno.nombre, "· código", alumno.codigo)

## Parte A · Actividad en clase (30 min)
Tu registrador mide desde −20 °C hasta un máximo propio, con un convertidor de N bits; guarda hasta 32 000 lecturas y
registra cada cierto intervalo. Vigila un horno cuya temperatura sube y baja con un periodo propio. Ejecuta la celda
para ver **tus** datos.

**Cómo se llena cada ejercicio.** Ejemplo con otros datos (no son los tuyos):
```
lsb_C  = RANGO_C / 2 ** N_BITS
SUST_1 = "50 / 2**6 = 0.781 °C"
PRED_1 = "las lecturas sólo podrán cambiar en saltos de unas ocho décimas de grado"
```

In [ ]:
T_MIN_C = -20.0
RANGO_C = parametro(alumno, "rango", 50, 120, 0)       # el registrador mide de T_MIN_C a T_MIN_C + RANGO_C
N_BITS = asignar(alumno, [8, 10, 12], salto=0)
MEMORIA = 32000                                        # lecturas
INTERVALO_S = asignar(alumno, [10, 30, 60, 120, 300], salto=1)
PERIODO_MIN = parametro(alumno, "periodo", 8, 30, 1)   # periodo del ciclo del horno, en minutos
print(f"Tus datos · convertidor de {N_BITS} bits · mide de {T_MIN_C:g} a {T_MIN_C + RANGO_C:g} °C (rango {RANGO_C:g} °C) · memoria {MEMORIA} lecturas")
print(f"           intervalo de registro {INTERVALO_S} s · periodo del horno {PERIODO_MIN} min")

### ★ Ejercicio 1 · ¿Qué tan fino mide? (3 pts)
**Para qué sirve:** la resolución es el cambio más pequeño que el registrador puede distinguir. Si es mayor que
la variación que te interesa, el registro no sirve.

El convertidor divide el rango en 2ᴺ escalones; un escalón (LSB) vale **LSB = rango / 2ᴺ**. En Python, la potencia
se escribe `2 ** N_BITS`.

In [ ]:
# ✎ 1a · La expresión en Python, con las variables de tus datos
lsb_C = None   # ✎ escribe aquí
# ✎ 1b · La sustitución con TUS números, el resultado y la unidad
SUST_1 = ""   # ✎ escribe aquí
# ✎ 1c · Lo que predice en la práctica, con tus palabras (al menos 10)
PRED_1 = ""   # ✎ escribe aquí

revisa(1, lsb_C, SUST_1, PRED_1, datos=[RANGO_C, N_BITS], unidades=['°C'])

### Ejercicio 2 · ¿Cuánto se equivoca al redondear? (2 pts)
**Para qué sirve:** es el error que agrega el registrador aunque el sensor fuera perfecto; entra en el presupuesto
de error de la medición.

Cada lectura se redondea al escalón más cercano. Piensa: si el valor real cae justo entre dos escalones, ¿qué
fracción de un LSB es lo más lejos que puede quedar? Escribe el error máximo en °C, a partir de tu `lsb_C`.

In [ ]:
# ✎ 2a · La expresión en Python, con las variables de tus datos
error_max_C = None   # ✎ escribe aquí
# ✎ 2b · La sustitución con TUS números, el resultado y la unidad
SUST_2 = ""   # ✎ escribe aquí
# ✎ 2c · Lo que predice en la práctica, con tus palabras (al menos 10)
PRED_2 = ""   # ✎ escribe aquí

revisa(2, error_max_C, SUST_2, PRED_2, datos=[lsb_C], unidades=['°C'])

### Ejercicio 3 · ¿Cuánto dura la memoria? (2 pts)
**Para qué sirve:** define cada cuánto hay que ir a descargar el registrador o si la campaña cabe completa.

La memoria se llena cuando ha guardado todas sus lecturas, una por intervalo. Calcula ese tiempo **en días**
(un día tiene 86 400 s).

In [ ]:
# ✎ 3a · La expresión en Python, con las variables de tus datos
duracion_dias = None   # ✎ escribe aquí
# ✎ 3b · La sustitución con TUS números, el resultado y la unidad
SUST_3 = ""   # ✎ escribe aquí
# ✎ 3c · Lo que predice en la práctica, con tus palabras (al menos 10)
PRED_3 = ""   # ✎ escribe aquí

revisa(3, duracion_dias, SUST_3, PRED_3, datos=[MEMORIA, INTERVALO_S], unidades=['día'])

### Ejercicio 4 · ¿Cada cuánto registrar como máximo? (2 pts)
**Para qué sirve:** si se registra muy espaciado, el registro inventa un ciclo que no existe (aliasing) y se toman
decisiones de mantenimiento con datos falsos.

El criterio de Nyquist pide **al menos dos registros por cada periodo** del proceso. Calcula el intervalo máximo en
minutos a partir de `PERIODO_MIN`.

In [ ]:
# ✎ 4a · La expresión en Python, con las variables de tus datos
intervalo_max_min = None   # ✎ escribe aquí
# ✎ 4b · La sustitución con TUS números, el resultado y la unidad
SUST_4 = ""   # ✎ escribe aquí
# ✎ 4c · Lo que predice en la práctica, con tus palabras (al menos 10)
PRED_4 = ""   # ✎ escribe aquí

revisa(4, intervalo_max_min, SUST_4, PRED_4, datos=[PERIODO_MIN], unidades=['min'])

La gráfica muestra el horno y lo que guardaría tu registrador en el límite de Nyquist.

In [ ]:
t = np.linspace(0, 3 * PERIODO_MIN, 600)
T = 180 + 40 * np.sin(2 * np.pi * t / PERIODO_MIN)
tm = np.arange(0, 3 * PERIODO_MIN, intervalo_max_min)
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(t, T, label="horno")
ax.plot(tm, 180 + 40 * np.sin(2 * np.pi * tm / PERIODO_MIN), "o", label=f"registro cada {intervalo_max_min:.1f} min")
ax.set_xlabel("tiempo (min)"); ax.set_ylabel("temperatura (°C)"); ax.legend()
titulo(ax, "Registro al límite de Nyquist", alumno)
firma(fig, alumno); plt.show()

### Cierre (1 pt)
Con tus números: si el registrador guardara una lectura **cada intervalo máximo del ejercicio 4**, ¿cuántos días
duraría la memoria? ¿Te conviene ese intervalo para una campaña de una semana? Escribe al menos 20 palabras e
incluye el número de días que calculaste.

In [ ]:
CIERRE = ""   # ✎ escribe aquí

revisa_texto('Cierre', CIERRE, {'el número de días que dura la memoria a ese intervalo':
             MEMORIA * intervalo_max_min * 60 / 86400})

## Parte B · Práctica (80 min)
**Objetivo:** comprobar con datos reales lo que predijiste en el ejercicio 2.

| Bloque | min | Qué haces |
|---|---|---|
| B1 · Datos | 10 | Descarga una semana de temperatura real de Guadalajara, hora por hora ([Open-Meteo](https://open-meteo.com/), CC BY 4.0). La semana depende de tu código |
| B2 · Registro | 20 | La registras con **tu** convertidor y comparas la señal real con la registrada |
| B3 · Error | 20 | Calculas el error máximo obtenido y lo comparas con el predicho |
| B4 · Conclusión | 20 | Escribes qué encontraste, con tus números |
| Tabla y captura | 10 | Descargas el CSV y lo importas en tu hoja de resultados |

In [ ]:
import json, urllib.request, datetime as dt
lunes = dt.date(2025, 1, 6) + dt.timedelta(weeks=int(alumno.codigo) % 50)
domingo = lunes + dt.timedelta(days=6)
url = ("https://archive-api.open-meteo.com/v1/archive?latitude=20.67&longitude=-103.35"
       f"&start_date={lunes}&end_date={domingo}&hourly=temperature_2m&timezone=America%2FMexico_City")
try:
    with urllib.request.urlopen(url, timeout=30) as resp:
        d = json.load(resp)["hourly"]
    fechas, T_real = d["time"], np.array(d["temperature_2m"], dtype=float)
    FUENTE = "Open-Meteo (ERA5), Guadalajara"
except Exception as e:
    print("⚠ No se pudo descargar de Open-Meteo:", type(e).__name__, e)
    print("⚠ Se usan datos SINTÉTICOS. Vuelve a ejecutar esta celda con internet antes de entregar.")
    h = np.arange(7 * 24)
    fechas = [f"hora {i}" for i in h]
    T_real = 20 + 7 * np.sin(2 * np.pi * (h - 9) / 24) + alumno.rng("ruido").normal(0, 0.6, h.size)
    FUENTE = "SINTÉTICOS (sin conexión)"
print(f"B1 · Semana del {lunes} al {domingo} · {len(T_real)} lecturas · fuente: {FUENTE}")

In [ ]:
# B2 · Registro con tu convertidor: −20 °C es la cuenta 0 y cada cuenta vale un LSB
cuentas = np.round((T_real - T_MIN_C) / lsb_C)
T_cuant = cuentas * lsb_C + T_MIN_C
error = T_cuant - T_real

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(T_real, label="real"); ax.step(range(len(T_cuant)), T_cuant, where="mid", label=f"registrada, {N_BITS} bits")
ax.set_ylabel("T (°C)"); ax.set_xlabel("lectura (h)"); ax.legend()
titulo(ax, "Temperatura real y registrada", alumno)
firma(fig, alumno); plt.show()

### B3 · Error máximo obtenido
El arreglo `error` tiene la diferencia entre lo registrado y lo real en cada hora. El error máximo es el **mayor
valor absoluto** de ese arreglo. Pistas: `np.abs(...)` da el valor absoluto y `np.max(...)` el mayor.

In [ ]:
error_obtenido_C = None   # ✎ escribe aquí

completa(error_obtenido_C, "el error máximo obtenido")
RESULTADOS = {"lsb_C": lsb_C, "error_predicho_C": error_max_C, "error_obtenido_C": float(error_obtenido_C)}
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(error, label="error de cada lectura")
ax.axhline(error_max_C, ls="--", c="r", label="± error predicho"); ax.axhline(-error_max_C, ls="--", c="r")
ax.set_ylabel("error (°C)"); ax.set_xlabel("lectura (h)"); ax.legend(loc="upper right")
titulo(ax, "Error de cuantización", alumno)
firma(fig, alumno, RESULTADOS); plt.show()
print(f"Error predicho {error_max_C:.4f} °C · error obtenido {error_obtenido_C:.4f} °C")

### B4 · Conclusión
¿El error obtenido quedó dentro del predicho? ¿Por qué? Escribe al menos 25 palabras e incluye tus dos valores.

In [ ]:
CONCLUSION_B = ""   # ✎ escribe aquí

revisa_texto('Conclusión de la práctica', CONCLUSION_B,
             {'el error predicho': error_max_C, 'el error obtenido': error_obtenido_C}, minimo=25)

### Tabla y captura · CSV para tu hoja de resultados
La celda descarga `S0_<código>.csv`. En tu hoja de resultados: abre la pestaña **CSV** → **Archivo → Importar →
Subir** → elige el archivo → **Reemplazar la hoja actual**. La pestaña Resultados se llena sola.

In [ ]:
ruta = f"S0_{alumno.codigo}.csv"
exportar_csv(ruta, alumno, RESULTADOS,
             {"fecha": fechas, "T_real_C": T_real, "T_registrada_C": T_cuant, "error_C": error},
             extra={"fuente": FUENTE, "semana": f"{lunes} a {domingo}"})
print("Archivo listo:", ruta, "· huella", huella(alumno, RESULTADOS))
try:
    from google.colab import files
    files.download(ruta)
except ImportError:
    pass

## Cierre del cuaderno
Antes de entregar: **Entorno de ejecución → Reiniciar y ejecutar todo**. Todo debe salir ✔.

In [ ]:
for k, ok in REGISTRO.items():
    print(("✔" if ok else "✘"), k if isinstance(k, str) else f"Ejercicio {k}")
print(f"\n{alumno.nombre} · {alumno.codigo} · {SESION} · {ahora()} · fuente: {FUENTE} · huella {huella(alumno, RESULTADOS)}")